# Tier 1: Empirical Risk Forecasting

Builds a historical lookup table answering: for a given hotspot, day of
week, and hour -- what's the empirical violation risk? This is a
climatology-style baseline forecast (every number traces back to real
historical counts, no extrapolation/model guessing).

**Prerequisite:** run `2.ipynb` first (clustering + scoring). This notebook
reads `cluster_summary_checkpoint.csv` and `hotspots_details.json`, both
produced by that notebook, and adds a `risk_by_day_hour` field to each
hotspot's entry in `hotspots_details.json` -- existing fields
(`vehicle_composition`, `top_violations`, `temporal_distribution`,
`operational_metrics`) are left untouched. `hotspots_summary.json` is not
modified at all.

In [1]:
import pandas as pd
import numpy as np
import json

In [2]:
cluster_summary = pd.read_csv("../../data/processed/cluster_summary_checkpoint.csv")
df_clusters = pd.read_csv("../../data/processed/parking_violations_exploded_with_clusters.csv")

print(f"Loaded {len(cluster_summary)} clusters, {len(df_clusters)} violation-level rows")

Loaded 300 clusters, 323431 violation-level rows


In [ ]:
df_clusters['cdt_day_of_week'] = pd.to_datetime(df_clusters['created_datetime']).dt.day_name()

risk_table = (
    df_clusters
    .groupby(['cluster_id', 'cdt_day_of_week', 'cdt_hour_bucket'])
    .agg(violation_count=('id', 'nunique'))
    .reset_index()
)

cluster_max = risk_table.groupby('cluster_id')['violation_count'].transform('max')
risk_table['risk_score'] = (risk_table['violation_count'] / cluster_max * 100).round(1)

print(f"Risk table shape: {risk_table.shape}")
risk_table.head(10)

Risk table shape: (13913, 5)


,cluster_id,cdt_day_of_week,cdt_hour_bucket,violation_count,risk_score
0,0,Friday,0,4,7.5
1,0,Friday,4,2,3.8
2,0,Friday,5,11,20.8
3,0,Friday,8,2,3.8
4,0,Friday,9,34,64.2
5,0,Friday,10,12,22.6
6,0,Friday,11,1,1.9
7,0,Friday,12,2,3.8
8,0,Friday,13,1,1.9
9,0,Monday,3,3,5.7


In [ ]:
def predict_risk(cluster_id, day_of_week, hour, risk_table=risk_table):

    match = risk_table[
        (risk_table['cluster_id'] == cluster_id) &
        (risk_table['cdt_day_of_week'] == day_of_week) &
        (risk_table['cdt_hour_bucket'] == hour)
    ]
    if not match.empty:
        return {
            "cluster_id": int(cluster_id), "day_of_week": day_of_week, "hour": int(hour),
            "risk_score": float(match['risk_score'].values[0]),
            "historical_count": int(match['violation_count'].values[0]),
            "basis": "historical_match"
        }
    return {
        "cluster_id": int(cluster_id), "day_of_week": day_of_week, "hour": int(hour),
        "risk_score": 0.0, "historical_count": 0,
        "basis": "no_historical_data_for_this_slot"
    }

top_cluster_id = int(cluster_summary.sort_values('congestion_impact_score', ascending=False).iloc[0]['cluster_id'])
top_peak_hour = int(cluster_summary[cluster_summary['cluster_id'] == top_cluster_id]['primary_peak_hour'].values[0])

print(f"Top hotspot is cluster {top_cluster_id}, known primary peak hour: {top_peak_hour}:00")
print("At peak hour, Monday:", predict_risk(top_cluster_id, "Monday", top_peak_hour))
print("At 3am (off-peak), Monday:", predict_risk(top_cluster_id, "Monday", 3))

Top hotspot is cluster 2, known primary peak hour: 9:00
At peak hour, Monday: {'cluster_id': 2, 'day_of_week': 'Monday', 'hour': 9, 'risk_score': 62.3, 'historical_count': 876, 'basis': 'historical_match'}
At 3am (off-peak), Monday: {'cluster_id': 2, 'day_of_week': 'Monday', 'hour': 3, 'risk_score': 22.7, 'historical_count': 319, 'basis': 'historical_match'}


## Export — merge into existing `hotspots_details.json`

Adds a `risk_by_day_hour` key to each hotspot's entry. Does **not** touch
`hotspots_summary.json` or any existing field in `hotspots_details.json` --
purely additive, so anything already built against those files keeps
working unchanged.

In [ ]:
DETAILS_PATH = "../../data/processed/hotspots_details.json"

with open(DETAILS_PATH, "r") as f:
    hotspots_details = json.load(f)

n_merged = 0
n_no_match = 0

for cid, group in risk_table.groupby('cluster_id'):
    cid_str = str(int(cid))
    risk_by_day = {}
    for day, day_group in group.groupby('cdt_day_of_week'):
        risk_by_day[day] = {
            str(int(row['cdt_hour_bucket'])): {
                "risk_score": row['risk_score'],
                "historical_count": int(row['violation_count'])
            }
            for _, row in day_group.iterrows()
        }
    if cid_str in hotspots_details:
        hotspots_details[cid_str]["risk_by_day_hour"] = risk_by_day
        n_merged += 1
    else:
        n_no_match += 1

print(f"Merged risk_by_day_hour into {n_merged} hotspot entries")
if n_no_match:
    print(f"WARNING: {n_no_match} clusters in risk_table had no matching entry "
          f"in hotspots_details.json -- check that both came from the same run")

with open(DETAILS_PATH, "w") as f:
    json.dump(hotspots_details, f, indent=2)

print(f"Saved updated {DETAILS_PATH}")

risk_table.to_csv("../../data/processed/cluster_risk_table_debug.csv", index=False)

Merged risk_by_day_hour into 300 hotspot entries
Saved updated ../../data/processed/hotspots_details.json
